## Parte A

In [1]:
import json
import pandas as pd
import numpy as np

### Carregamento dos dados em um dataframe e extração da taxa de câmbio

In [2]:
with open("../dados/dados_nivel_1.json", "r", encoding="utf-8") as info:
    dados = json.load(info)

taxa_cambio = dados["taxa_cambio_usd_brl"]
df = pd.DataFrame(dados["operacoes"])
df

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 1.5 KB


### Limpeza dos dados

#### Problemas identificados:

- **Tipo da coluna de data**  
Problema: O tipo string torna difícil a filtrar os dados por um determinado período de tempo  
Solução: Conversão do tipo para datetime

- **Tipo da coluna valor**  
Problema: O tipo inteiro pode causar perda de informação já que é comum transferências e pagamentos não serem valores inteiros. Além disso a taxa de câmbio normalmente não é um valor inteiro.  
Solução: Conversão do tipo para float

- **Tipos das colunas moedas, canal e tipo**  
Problema: Considerando que possuem poucas variações de valor para essas variáveis, por exemplo, a coluna canal pode ser "pix", "ted", "cartao", "boleto", "especie", o tipo string não é o mais vantajoso.  
Solução: Utilizar variáveis categóricas delimitando os possíveis valores desses campos. Assim é possível otimizar memória operações como agrupamento e filtros. Além disso, evita criação de novas categorias por erros de digitação. Seria ideal definir as categorias antes da criação da tabela, mas nesse contexto, será feita a criação com base nos valores existentes.


In [4]:
# Aplicando soluções

df["data"] = pd.to_datetime(df["data"], format="%Y-%m-%d")
df["valor"] = df["valor"].astype(float)
df["moeda"] = df["moeda"].astype("category")
df["canal"] = df["canal"].astype("category")
df["tipo"] = df["tipo"].astype("category")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id           20 non-null     str           
 1   cliente_id   20 non-null     str           
 2   data         19 non-null     datetime64[us]
 3   valor        20 non-null     float64       
 4   moeda        20 non-null     category      
 5   canal        20 non-null     category      
 6   tipo         20 non-null     category      
 7   contraparte  20 non-null     str           
 8   observacao   20 non-null     str           
dtypes: category(3), datetime64[us](1), float64(1), str(4)
memory usage: 1.2 KB


In [5]:
print(df["canal"].cat.categories,
df["moeda"].cat.categories,
df["tipo"].cat.categories)

Index(['boleto', 'cartao', 'especie', 'pix', 'ted'], dtype='str') Index(['BRL', 'USD'], dtype='str') Index(['deposito', 'pagamento', 'transferencia_enviada',
       'transferencia_recebida'],
      dtype='str')


### Normalização dos valores para BRL

In [6]:
condition = df["moeda"] == "USD"

df.loc[condition, "valor"] = df.loc[condition, "valor"] * taxa_cambio
df.loc[condition, "moeda"] = "BRL"

df

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100.0,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300.0,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800.0,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300.0,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900.0,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000.0,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100.0,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,


### Agregações

In [7]:
# Volume total transacionado por cliente

df_volume_cliente = df.groupby("cliente_id")["valor"].sum()
df_volume_cliente

cliente_id
CLI-A-1    57500.0
CLI-A-2    52900.0
CLI-A-3    65700.0
CLI-A-4    79500.0
CLI-A-5    16900.0
CLI-A-6    10200.0
Name: valor, dtype: float64

In [8]:
# Quantidade de operações por canal

df_operacoes_canal = df.groupby("canal")["id"].count()
df_operacoes_canal

canal
boleto     3
cartao     2
especie    1
pix        9
ted        5
Name: id, dtype: int64

### Criação de Flags

In [9]:
# Fracionamento

# Operações que não atingem R$ 20.000
df_menor_20k = df[df["valor"] < 20000]

# Calcula valor e número de operações por cliente por data
df_fracionamento = df_menor_20k.groupby(["cliente_id", "data"])["valor"].agg(
    qtd_operacoes="count",
    soma="sum"
).reset_index()

# Criação da flag de fracionamento
df_fracionamento["fracionamento"] = (
    (df_fracionamento["qtd_operacoes"] >= 3) &
    (df_fracionamento["soma"] > 50000)
)

df_fracionamento

,cliente_id,data,qtd_operacoes,soma,fracionamento
0,CLI-A-1,2026-03-09,3,54200.0,True
1,CLI-A-1,2026-03-21,1,3300.0,False
2,CLI-A-3,2026-03-05,4,65700.0,True
3,CLI-A-4,2026-03-03,1,3800.0,False
4,CLI-A-4,2026-03-11,1,5100.0,False
5,CLI-A-4,2026-03-18,1,5800.0,False
6,CLI-A-5,2026-03-07,1,2900.0,False
7,CLI-A-5,2026-03-16,1,7000.0,False
8,CLI-A-5,2026-03-26,1,2700.0,False
9,CLI-A-6,2026-03-12,1,8800.0,False


In [10]:
# Valor atípico

# Calcula quantidade de operaçÕes e mediana por cliente
metricas = df.groupby("cliente_id")["valor"].agg(
    qtd_operacoes="count",
    mediana="median"
)

print(metricas)

# Criação da flag de valor atípico
df["val_atipico"] = (
    (df["cliente_id"].map(metricas["qtd_operacoes"]) >= 4) &
    (df["valor"] > 5 * df["cliente_id"].map(metricas["mediana"]))
)

df

            qtd_operacoes  mediana
cliente_id                        
CLI-A-1                 4  17700.0
CLI-A-2                 2  26450.0
CLI-A-3                 4  16650.0
CLI-A-4                 4   5450.0
CLI-A-5                 4   3600.0
CLI-A-6                 2   5100.0


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,val_atipico
0,OP-0001,CLI-A-1,2026-03-09,18100.0,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False
1,OP-0002,CLI-A-1,2026-03-09,17300.0,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False
2,OP-0003,CLI-A-1,2026-03-09,18800.0,BRL,ted,transferencia_enviada,Beta Servicos ME,,False
3,OP-0004,CLI-A-1,2026-03-21,3300.0,BRL,boleto,pagamento,Gama Distribuidora,,False
4,OP-0005,CLI-A-2,2026-03-14,25900.0,BRL,ted,transferencia_enviada,Delta Transportes,,False
5,OP-0006,CLI-A-2,2026-03-14,27000.0,BRL,ted,transferencia_enviada,Delta Transportes,,False
6,OP-0007,CLI-A-3,2026-03-05,17200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False
7,OP-0008,CLI-A-3,2026-03-05,15200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False
8,OP-0009,CLI-A-3,2026-03-05,16100.0,BRL,pix,transferencia_enviada,Zeta Importacao,,False
9,OP-0007,CLI-A-3,2026-03-05,17200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False


### Validação das regras

In [12]:
# Validação da regra 1

#cliente 1: tem 3 transações no mesmo dia, menores que 20.000 que somam mais de 50.000
#cliente 4: tem mais de 3 transações que somam mais de 50.000 mas são em dias diferentes e uma transação é acima de 20.000

resultado_cliente_1 = df_fracionamento.loc[df_fracionamento["cliente_id"] == "CLI-A-1", "fracionamento"].iloc[0]
resultado_cliente_4 = df_fracionamento.loc[df_fracionamento["cliente_id"] == "CLI-A-4", "fracionamento"].iloc[0]

print(df_fracionamento)
print(f"\nCliente 1 flagado: {resultado_cliente_1} -> esperado: True")
print(f"Cliente 4 flagado: {resultado_cliente_4} -> esperado: False")

assert resultado_cliente_1 == True, "ERRO: cliente 1 deveria ter sido flagado"
assert resultado_cliente_4 == False, "ERRO: cliente 4 não deveria ter sido flagado"

   cliente_id       data  qtd_operacoes     soma  fracionamento
0     CLI-A-1 2026-03-09              3  54200.0           True
1     CLI-A-1 2026-03-21              1   3300.0          False
2     CLI-A-3 2026-03-05              4  65700.0           True
3     CLI-A-4 2026-03-03              1   3800.0          False
4     CLI-A-4 2026-03-11              1   5100.0          False
5     CLI-A-4 2026-03-18              1   5800.0          False
6     CLI-A-5 2026-03-07              1   2900.0          False
7     CLI-A-5 2026-03-16              1   7000.0          False
8     CLI-A-5 2026-03-26              1   2700.0          False
9     CLI-A-6 2026-03-12              1   8800.0          False
10    CLI-A-6 2026-03-28              1   1400.0          False

Cliente 1 flagado: True -> esperado: True
Cliente 4 flagado: False -> esperado: False
